In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


In [2]:
# Cell 3 — Verify uploaded inputs
from pathlib import Path

RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
utm_academic_regulations_16th_ed.pdf bytes: 779259
utm_genai_guidelines.pdf bytes: 1409304
Upload verified. Select your relevant input(s) for Step A3.


In [3]:
# Cell A3 — Inspect the two selected PDFs
!pip -q install pypdf

from pathlib import Path
from pypdf import PdfReader

RAW_DIR = Path("/content/week3_project/data/raw")

SOURCES = {
    "S01": "utm_genai_guidelines.pdf",
    "S02": "utm_academic_regulations_16th_ed.pdf",
}

KEYWORDS = {
    "S01": ["generative", "assessment", "lecturer"],
    "S02": ["misconduct", "plagiarism", "examination"],
}

for sid, name in SOURCES.items():
    path = RAW_DIR / name
    assert path.is_file(), f"Missing file: {path}"

    reader = PdfReader(str(path))
    pages = [(page.extract_text() or "") for page in reader.pages]
    lengths = [len(p.strip()) for p in pages]
    empty_pages = [i + 1 for i, n in enumerate(lengths) if n < 20]

    print("=" * 60)
    print(sid, "|", name)
    print("Pages:", len(pages))
    print("Total characters:", sum(lengths))
    print("Average characters per page:", round(sum(lengths) / len(pages)))
    print("Pages with almost no text:", empty_pages if empty_pages else "none")
    print("PDF metadata title:", reader.metadata.title if reader.metadata else None)

    for kw in KEYWORDS[sid]:
        hits = [i + 1 for i, p in enumerate(pages) if kw in p.lower()]
        print(f"Pages mentioning '{kw}':", len(hits), "| first pages:", hits[:8])

    first_text_page = next((i for i, n in enumerate(lengths) if n >= 20), 0)
    mid = len(pages) // 2
    print(f"\nSample 1 (page {first_text_page + 1}):")
    print(repr(pages[first_text_page].strip()[:400]))
    print(f"\nSample 2 (page {mid + 1}):")
    print(repr(pages[mid].strip()[:400]))

S01 | utm_genai_guidelines.pdf
Pages: 17
Total characters: 15189
Average characters per page: 893
Pages with almost no text: none
PDF metadata title: None
Pages mentioning 'generative': 5 | first pages: [1, 2, 3, 5, 17]
Pages mentioning 'assessment': 3 | first pages: [5, 6, 12]
Pages mentioning 'lecturer': 1 | first pages: [4]

Sample 1 (page 1):
'GUIDELINES FOR\nTHE USE OF GENERATIVE \nARTIFICIAL INTELLIGENCE \nIN TEACHING AND \nLEARNING'

Sample 2 (page 9):
'8  \n2.3.7 Any writi ng and works in teaching and learning activities that do not \ndisclose the use of GAI (if applicable) can be categori sed as plagiarism \naccording to University Academic Regulations. \n \n2.4 Research and Publication  \nFor research and publication, the following aspects need to be considered: \n \n \n2.4.1 The use of services and tools such as proofreaders, proofreading tools, \nspel'
S02 | utm_academic_regulations_16th_ed.pdf
Pages: 81
Total characters: 105112
Average characters per page: 1298
Pages with 

In [4]:
# Cell A3b — Check the garbled pages and the misconduct pages in S02
from pathlib import Path
from pypdf import PdfReader

path = Path("/content/week3_project/data/raw/utm_academic_regulations_16th_ed.pdf")
reader = PdfReader(str(path))
pages = [(p.extract_text() or "") for p in reader.pages]

garbled = [i + 1 for i, p in enumerate(pages) if "/gid" in p]
print("Pages containing unreadable '/gid' codes:", garbled if garbled else "none")
print("Count:", len(garbled), "of", len(pages))

for pg in [8, 59, 60, 75]:
    print(f"\n--- PDF page {pg} ---")
    print(repr(pages[pg - 1].strip()[:600]))

Pages containing unreadable '/gid' codes: [4]
Count: 1 of 81

--- PDF page 8 ---
'vii\nContents Contents\n32. Announcement of  the Academic Status 39\n33. Dean’s Honour List 40\n34. Graduation Requirements 40\n35. UTM Professional Skill Certificate (UTM PSC) 41\n36. Posthumous Award 41\nSECTION 7 ACADEMIC MISCONDUCT 42\nArticle  \n37. Academic Deviation 42-43\n38. Misconduct During Final Examination 44\n39. Implication of  Misconduct 44\nSECTION 8 WITHDRAWAL FROM THE UNIVERSITY 45\nArticle  \n40. Withdrawal from the University 45\nSECTION 9 GENERAL PROVISION,\n  ENFORCEMENT , AND\n APPLICATION OF REGULATIONS  46\nArticle  \n41. General Provision 46\n42. Enforcement and Application of  Regulati'

--- PDF page 59 ---
'42\nAcademic Regulations\nSECTION 7\nACADEMIC MISCONDUCT\nArticle 37: Academic Deviation\n(1) Academic Misconduct refers to violations of  academic integrity that \nmay result in disciplinary action if  the student is found guilty . Academic \nmisconduct includes cheating d

In [5]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError("Select exactly one source. Set relative_path to an exact file under RAW_DIR.")
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

print("Cell 4 completed successfully.")

Cell 4 completed successfully.


In [6]:
# CELL 4A — COMMON LINEAGE REGISTRY
from pathlib import Path

lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [7]:
# Cell 5A — Compare fixed-size vs page-aware chunking (S01 and S02)
!pip -q install pypdf
import re
from pypdf import PdfReader

SOURCES = {
    "S01": "utm_genai_guidelines.pdf",
    "S02": "utm_academic_regulations_16th_ed.pdf",
}
KEY_PHRASE = {"S01": "plagiarism", "S02": "Article 37"}
CHUNK_SIZE = 500
MIN_PAGE_CHARS = 20

def clean_text(s):
    return re.sub(r"[ \t]+", " ", s).strip()

def read_pages(path):
    reader = PdfReader(str(path))
    pages, skipped = [], []
    for i, page in enumerate(reader.pages, 1):
        t = clean_text(page.extract_text() or "")
        if len(t) < MIN_PAGE_CHARS or "/gid" in t:
            skipped.append(i)
        else:
            pages.append((i, t))
    return pages, skipped

def fixed_chunks(s, size):
    return [s[i:i + size] for i in range(0, len(s), size)]

def pack(units, size, sep):
    chunks, cur = [], ""
    for u in units:
        u = u.strip()
        if not u:
            continue
        if len(u) > size:
            if cur:
                chunks.append(cur)
                cur = ""
            chunks.extend(u[i:i + size] for i in range(0, len(u), size))
        elif len(cur) + len(u) + (len(sep) if cur else 0) <= size:
            cur += (sep if cur else "") + u
        else:
            chunks.append(cur)
            cur = u
    if cur:
        chunks.append(cur)
    return chunks

def structure_chunks(page_text, size):
    units = []
    for para in re.split(r"\n\s*\n", page_text):
        para = para.strip()
        if not para:
            continue
        if len(para) <= size:
            units.append(para)
        else:
            units.extend(pack(para.split("\n"), size, "\n"))
    return pack(units, size, "\n\n")

def join_with_offsets(pages):
    text, starts = "", []
    for pg, t in pages:
        starts.append((len(text), pg))
        text += t + "\n"
    return text, starts

def page_of(pos, starts):
    pg = starts[0][1]
    for s, p in starts:
        if s <= pos:
            pg = p
        else:
            break
    return pg

def stats(label, items):
    lens = [len(c) for _, c in items]
    print(f"  {label}: chunks={len(lens)}, avg length={round(sum(lens)/len(lens))}, "
          f"min={min(lens)}, chunks under 100 chars={sum(1 for n in lens if n < 100)}")

prep = {}
for sid, name in SOURCES.items():
    path = select_source(relative_path=name)
    pages, skipped = read_pages(path)
    assert pages, "No usable text found"
    text, starts = join_with_offsets(pages)

    raw_fixed = fixed_chunks(text, CHUNK_SIZE)
    fixed, cross, midword = [], 0, 0
    for i, ch in enumerate(raw_fixed):
        s = i * CHUNK_SIZE
        e = s + len(ch) - 1
        p1, p2 = page_of(s, starts), page_of(e, starts)
        fixed.append((p1, ch))
        if p1 != p2:
            cross += 1
        if i + 1 < len(raw_fixed) and ch[-1].isalnum() and raw_fixed[i + 1][0].isalnum():
            midword += 1

    page_aware = []
    for pg, t in pages:
        for ch in structure_chunks(t, CHUNK_SIZE):
            page_aware.append((pg, ch))

    prep[sid] = {"path": path, "name": name, "fixed": fixed, "page_aware": page_aware}

    print("=" * 60)
    print(sid, name)
    print("Pages used:", len(pages), "| Pages skipped (empty or unreadable):", skipped)
    stats("A fixed-size", fixed)
    print(f"     fixed chunks that mix two pages: {cross} of {len(fixed)}")
    print(f"     fixed chunks cut inside a word: {midword}")
    stats("B page-aware", page_aware)
    print("     page-aware chunks that mix two pages: 0 (by design)")

    phrase = KEY_PHRASE[sid]
    a = next(((p, c) for p, c in fixed if phrase.lower() in c.lower()), None)
    b = next(((p, c) for p, c in page_aware if phrase.lower() in c.lower()), None)
    print(f"\n  Sample A containing '{phrase}' (starts on PDF page {a[0] if a else None}):")
    print("  ", repr(a[1][:300]) if a else "not found")
    print(f"\n  Sample B containing '{phrase}' (PDF page {b[0] if b else None}):")
    print("  ", repr(b[1][:300]) if b else "not found")

S01 utm_genai_guidelines.pdf
Pages used: 17 | Pages skipped (empty or unreadable): []
  A fixed-size: chunks=31, avg length=489, min=157, chunks under 100 chars=0
     fixed chunks that mix two pages: 15 of 31
     fixed chunks cut inside a word: 15
  B page-aware: chunks=44, avg length=338, min=56, chunks under 100 chars=3
     page-aware chunks that mix two pages: 0 (by design)

  Sample A containing 'plagiarism' (starts on PDF page 4):
   'y new knowledge and \nskills, and be prepared to face changes and challenges \nin their tasks throughout their service period. \nPlagiarism The use of ideas, words, and/or work of others without \nacknowledging the original source. \nCourse One unit of instruction and teaching of a program also \nknown as'

  Sample B containing 'plagiarism' (PDF page 4):
   'the enhancement of individual competencies enables\nindividuals to understand, apply new knowledge and\nskills, and be prepared to face changes and challenges\nin their tasks throughout their

In [8]:
# Cell 5A2 — Export chosen chunks, validate (B3), register lineage (B4.1)
import json

CHOSEN_METHOD = "page-aware"   # change to "fixed" only if your evidence supports it
if CHOSEN_METHOD == "page-aware":
    key, approach = "page_aware", "page-aware: paragraph then line packing, 500 chars, chunks never cross pages"
elif CHOSEN_METHOD == "fixed":
    key, approach = "fixed", "fixed-size: 500 chars over whole document; page = start page of chunk"
else:
    raise ValueError("Choose 'page-aware' or 'fixed'")

records = []
for sid, d in prep.items():
    sha = source_hash(d["path"])
    for n, (pg, chunk) in enumerate(d[key], 1):
        records.append({
            "record_id": f"{sid}-{n:04d}",
            "source_id": sid,
            "text": chunk,
            "source": str(d["path"].relative_to(RAW_DIR)),
            "source_sha256": sha,
            "processing": approach,
            "pdf_page": pg,   # PDF page index, not the printed page number
        })

prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
with prepared_path.open("w", encoding="utf-8") as out:
    for row in records:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

# B3 basic output check
assert prepared_path.exists(), "Prepared output missing."
assert prepared_path.stat().st_size > 0, "Prepared output is empty."
with prepared_path.open("r", encoding="utf-8") as fh:
    exported = [json.loads(line) for line in fh if line.strip()]
assert len(exported) == len(records), "Exported record count does not match"
assert all(r["text"].strip() for r in exported), "Empty chunk found"
assert all(isinstance(r["pdf_page"], int) for r in exported), "Missing page number"
print("Output:", prepared_path, "Size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")

for sid in prep:
    rows = [r for r in exported if r["source_id"] == sid]
    print(sid, "records:", len(rows), "| first record:", {k: rows[0][k] for k in ("record_id", "pdf_page", "processing")})
    print("   text sample:", repr(rows[0]["text"][:150]))

# B4.1 lineage registration (one entry per source)
for sid, d in prep.items():
    register_lineage(source=d["path"], output=prepared_path, processing=approach)
print("Registered entries:", len(lineage_entries))

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 208453 bytes
Validation: PASS
S01 records: 44 | first record: {'record_id': 'S01-0001', 'pdf_page': 1, 'processing': 'page-aware: paragraph then line packing, 500 chars, chunks never cross pages'}
   text sample: 'GUIDELINES FOR\nTHE USE OF GENERATIVE \nARTIFICIAL INTELLIGENCE \nIN TEACHING AND \nLEARNING'
S02 records: 249 | first record: {'record_id': 'S02-0001', 'pdf_page': 5, 'processing': 'page-aware: paragraph then line packing, 500 chars, chunks never cross pages'}
   text sample: 'First Edition 2026 \n© MOHD SHAFRY MOHD RAHIM & INTAN ZAURAH MAT DARUS 2026'
Registered: utm_genai_guidelines.pdf -> prepared_text.jsonl
Registered: utm_academic_regulations_16th_ed.pdf -> prepared_text.jsonl
Registered entries: 2


In [10]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
assert lineage_entries, (
    "No lineage entries registered. "
    "Register at least one selected pathway first."
)

lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Not verified",
        "access_rights": "Not verified",
        "status": "Prepared; quality review pending"
    })

manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 2
Saved to: /content/week3_project/data/week3_manifest.json


In [11]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
import json

manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."

with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)

saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), (
    "Saved lineage count does not match registration."
)

for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], (
        f"Source checksum mismatch: {source}"
    )
    assert source_hash(output) == record["prepared_sha256"], (
        f"Prepared output checksum mismatch: {output}"
    )

print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 2
Manifest: /content/week3_project/data/week3_manifest.json
